# 01 — Télécharger et extraire Defectors v2

Notebook Google Colab pour le projet Bug Predictor. Exécuter les cellules dans l’ordre.

## Objectif et précautions
Ce notebook télécharge Defectors v2 depuis Zenodo, vérifie la taille et le MD5 publiés, puis extrait uniquement `line_bug_prediction_splits/time/{train,val,test}.parquet.gzip` dans Google Drive. L’archive complète pèse environ 2,2 Go. Le split `test` est conservé pour audit seulement; ne l’utilisez pas pour entraîner, régler ou choisir un seuil. Le code source du dataset est traité comme donnée et n’est jamais exécuté.

In [ ]:
!pip -q install pyarrow pandas requests

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib, json, os, shutil, time, zipfile
import requests

ROOT = Path('/content/drive/MyDrive/BugPredictor/defectors_v2')
RAW_DIR = ROOT / 'raw' / 'line_bug_prediction_splits' / 'time'
RAW_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = Path('/content/defectors-v2.zip')
URL = 'https://zenodo.org/records/7708984/files/defectors.zip?download=1'
EXPECTED_MD5 = '7d6bab4dd3f12ecd76b8d8a8bc785b13'
EXPECTED_BYTES = 2223303371
SPLITS = ('train', 'val', 'test')

### Télécharger et contrôler l’archive
Si Colab interrompt le téléchargement, relancez la cellule. La somme MD5 est contrôlée avant extraction. La vérification vise l’archive officielle retenue, pas une copie trouvée ailleurs.

In [ ]:
def md5_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

if not all((RAW_DIR / f'{s}.parquet.gzip').exists() for s in SPLITS):
    free = shutil.disk_usage('/content').free
    print(f'Espace libre temporaire Colab : {free / 1024**3:.1f} Gio')
    if free < 3 * 1024**3:
        raise RuntimeError('Espace temporaire insuffisant : il faut au moins 3 Gio libres pour télécharger l’archive de 2,22 Go.')
    if not ZIP_PATH.exists() or ZIP_PATH.stat().st_size != EXPECTED_BYTES:
        ZIP_PATH.unlink(missing_ok=True)
        with requests.get(URL, stream=True, timeout=(30, 180)) as response:
            response.raise_for_status()
            downloaded = 0
            with open(ZIP_PATH, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
                        if downloaded % (128 * 1024 * 1024) < 8 * 1024 * 1024:
                            print(f'Téléchargé : {downloaded / 1024**3:.2f} Gio')
    if ZIP_PATH.stat().st_size != EXPECTED_BYTES:
        raise RuntimeError(f'Taille inattendue: {ZIP_PATH.stat().st_size} octets; attendu {EXPECTED_BYTES}.')
    actual_md5 = md5_file(ZIP_PATH)
    print('MD5 :', actual_md5)
    if actual_md5 != EXPECTED_MD5:
        raise RuntimeError('Le MD5 ne correspond pas à celui documenté pour Defectors v2; ne pas extraire cette archive.')
else:
    print('Les trois partitions existent déjà dans Drive; téléchargement ignoré.')

### Extraire seulement les partitions line-level temporelles

In [ ]:
if not all((RAW_DIR / f'{s}.parquet.gzip').exists() for s in SPLITS):
    if not ZIP_PATH.exists():
        raise FileNotFoundError('Archive absente: relancez la cellule de téléchargement.')
    with zipfile.ZipFile(ZIP_PATH) as archive:
        names = archive.namelist()
        selected = {}
        for split in SPLITS:
            suffix = f'line_bug_prediction_splits/time/{split}.parquet.gzip'
            matches = [name for name in names if name == suffix or name.endswith('/' + suffix)]
            if len(matches) != 1:
                candidates = [name for name in names if 'line_bug_prediction_splits/time/' in name and split in name]
                raise RuntimeError(f'Fichier attendu ambigu ou absent pour {split}. Candidats: {candidates[:10]}')
            selected[split] = matches[0]
        for split, member in selected.items():
            target = RAW_DIR / f'{split}.parquet.gzip'
            temporary = target.with_suffix(target.suffix + '.part')
            with archive.open(member) as src, open(temporary, 'wb') as dst:
                shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)
            temporary.replace(target)
            print(f'{split}: {target.stat().st_size:,} octets — {member}')

print('Partitions extraites :')
for split in SPLITS:
    print(split, RAW_DIR / f'{split}.parquet.gzip', (RAW_DIR / f'{split}.parquet.gzip').exists())

### Métadonnées de provenance
Le manifeste conserve l’origine, l’empreinte attendue, la date d’exécution et les fichiers extraits. Ce contrôle ne valide pas encore les labels des partitions; utilisez le notebook 02 pour l’audit complet.

In [ ]:
manifest = {
    'dataset': 'Defectors: A Large Scale Python Dataset for Defect Prediction',
    'version': 'v2',
    'doi': '10.5281/zenodo.7708984',
    'source_url': 'https://zenodo.org/records/7708984',
    'archive': 'defectors.zip',
    'archive_md5_expected': EXPECTED_MD5,
    'archive_bytes_expected': EXPECTED_BYTES,
    'selected_family': 'line_bug_prediction_splits/time',
    'partitions': {s: {'path': str(RAW_DIR / f'{s}.parquet.gzip'), 'bytes': (RAW_DIR / f'{s}.parquet.gzip').stat().st_size} for s in SPLITS},
    'created_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'test_policy': 'audit only; never train, tune, calibrate, or select thresholds on test'
}
(ROOT / 'raw_manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(manifest, indent=2, ensure_ascii=False))

Référence: [Zenodo — Defectors v2](https://zenodo.org/records/7708984). Conservez l’attribution et vérifiez les licences des projets sources avant toute redistribution.